# MRM Inventory Report Generator — Multi-Quarter
Scans folders for all inventory & validation quarterly files, concatenates them,
filters to a user-defined model list, and produces the final output report.
All records across quarters are preserved (no deduplication).

## 0. Configuration — Update These Before Running

In [ ]:
# =============================================================================
#  SECTION 0 — USER CONFIGURATION
# =============================================================================

# ----- Folder paths ----------------------------------------------------------
# Folder containing mrm_inventory_Q*.xlsx files
INVENTORY_FOLDER   = "data/inventory"

# Folder containing mrm_validation_inventory_Q*.xlsx files
VALIDATION_FOLDER  = "data/validation"

# Output file path
OUTPUT_FILE        = "mrm_output_report.xlsx"

# ----- Model filter list -----------------------------------------------------
# Paste your model IDs below (one per line inside the list).
# These should match the values in the 'Number' column of the inventory file.
MODEL_FILTER = [
    "MODEL-001",
    "MODEL-002",
    "MODEL-003",
    # ... add more IDs here ...
]

# ----- Column name mappings --------------------------------------------------
# Adjust if your actual file headers differ (run Cell 2 first to see them)
INV_MODEL_ID_COL   = "Number"             # Model ID col in inventory files
INV_MODEL_NAME_COL = "MRM Inventory Name" # Model name col in inventory files
INV_USE_STATUS_COL = "Use Status"         # Use Status col in inventory files

VAL_MODEL_ID_COL   = "Number (model id)"  # Model ID col in validation files (col 6)
VAL_TASK_ID_COL    = "Number"             # Validation Task ID col in validation files (col 1)
VAL_TYPE_COL       = "Validation Type"    # Validation type col (FSV / AR / LSV)
VAL_END_DATE_COL   = "Actual End Date"    # Actual End Date col in validation files

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

## 2. Load & Concatenate All Quarterly Files

In [ ]:
def load_folder(folder, label):
    """
    Reads all .xlsx files in `folder`, tags each row with its source filename
    and a quarter tag, then concatenates into one DataFrame.
    """
    folder = Path(folder)
    files  = sorted(folder.glob("*.xlsx"))
    
    if not files:
        raise FileNotFoundError(f"No .xlsx files found in: {folder.resolve()}")
    
    frames = []
    for f in files:
        df = pd.read_excel(f, dtype=str)
        df.columns = df.columns.str.strip()          # normalise headers
        df["_source_file"]    = f.name
        # Extract quarter tag from filename e.g. "Q1", "Q2" ... if present
        import re
        m = re.search(r"(Q\d)", f.name, re.IGNORECASE)
        df["_quarter"] = m.group(1).upper() if m else f.stem
        frames.append(df)
        print(f"  [{label}] Loaded {f.name:50s} → {len(df):>5} rows")
    
    combined = pd.concat(frames, ignore_index=True)
    print(f"  ─── {label} total rows: {len(combined)} across {len(files)} files\n")
    return combined

print("Loading INVENTORY files...")
df_inv = load_folder(INVENTORY_FOLDER, "INV")

print("Loading VALIDATION files...")
df_val = load_folder(VALIDATION_FOLDER, "VAL")

print("Inventory columns :", df_inv.columns.tolist())
print("Validation columns:", df_val.columns.tolist())

## 3. Normalise Join Keys & Apply Model Filter

In [ ]:
# Rename model ID columns to a common key
df_inv = df_inv.rename(columns={INV_MODEL_ID_COL: "_model_id"})
df_val = df_val.rename(columns={VAL_MODEL_ID_COL: "_model_id"})

df_inv["_model_id"] = df_inv["_model_id"].str.strip()
df_val["_model_id"] = df_val["_model_id"].str.strip()

# Normalise the filter list too
model_filter_set = {str(m).strip() for m in MODEL_FILTER if str(m).strip()}

df_inv_filtered = df_inv[df_inv["_model_id"].isin(model_filter_set)].copy()
df_val_filtered = df_val[df_val["_model_id"].isin(model_filter_set)].copy()

print(f"Inventory after filter : {len(df_inv_filtered):>5} rows  "
      f"({df_inv_filtered['_model_id'].nunique()} unique models)")
print(f"Validation after filter: {len(df_val_filtered):>5} rows  "
      f"({df_val_filtered['_model_id'].nunique()} unique models)")

# Warn if any requested model IDs are missing from the inventory
found    = set(df_inv_filtered["_model_id"].unique())
missing  = model_filter_set - found
if missing:
    print(f"\n⚠️  {len(missing)} model ID(s) from your filter list were NOT found in any inventory file:")
    for m in sorted(missing):
        print(f"   • {m}")

## 4. Separate Validation Records by Type (FSV / AR / LSV)

In [ ]:
def filter_by_type(df, keyword):
    mask = df[VAL_TYPE_COL].str.upper().str.contains(keyword.upper(), na=False)
    return df[mask].copy()

df_fsv = filter_by_type(df_val_filtered, "FSV")
df_ar  = filter_by_type(df_val_filtered, "AR")
df_lsv = filter_by_type(df_val_filtered, "LSV")

print(f"FSV rows : {len(df_fsv)}")
print(f"AR  rows : {len(df_ar)}")
print(f"LSV rows : {len(df_lsv)}")

## 5. Prepare Validation Subsets for Join (Rename Columns with Prefix)

In [ ]:
def prepare_val_subset(df, prefix):
    """
    Rename task-ID and date columns with a type prefix.
    ALL rows are kept — no deduplication.
    """
    df = df.copy()
    df[VAL_END_DATE_COL] = pd.to_datetime(df[VAL_END_DATE_COL], errors="coerce")
    df = df.sort_values(["_model_id", VAL_END_DATE_COL], ascending=[True, False])

    rename_map = {}
    if VAL_TASK_ID_COL in df.columns:
        rename_map[VAL_TASK_ID_COL]  = f"{prefix}_task_id"
    if VAL_END_DATE_COL in df.columns:
        rename_map[VAL_END_DATE_COL] = f"{prefix}_date"
    df = df.rename(columns=rename_map)

    # Also carry quarter tag so output row shows which quarter it came from
    df = df.rename(columns={"_quarter": f"{prefix}_quarter",
                             "_source_file": f"{prefix}_source"})

    keep = ["_model_id",
            f"{prefix}_task_id", f"{prefix}_date",
            f"{prefix}_quarter", f"{prefix}_source"]
    keep = [c for c in keep if c in df.columns]
    return df[keep]

df_fsv_prep = prepare_val_subset(df_fsv, "fsv")
df_ar_prep  = prepare_val_subset(df_ar,  "ar")
df_lsv_prep = prepare_val_subset(df_lsv, "lsv")

print(f"FSV prepared rows : {len(df_fsv_prep)}")
print(f"AR  prepared rows : {len(df_ar_prep)}")
print(f"LSV prepared rows : {len(df_lsv_prep)}")

## 6. Join & Build Output Rows
Inventory rows are the base. Each validation type is left-joined (all rows kept).
Where a model has multiple FSV/AR/LSV records, it will appear on multiple output rows.

In [ ]:
# Base: one row per (model, quarter) from inventory
inv_cols = ["_model_id", "_quarter", "_source_file",
            INV_MODEL_NAME_COL, INV_USE_STATUS_COL]
inv_cols = [c for c in inv_cols if c in df_inv_filtered.columns]

df_base = df_inv_filtered[inv_cols].drop_duplicates().copy()

# Outer merge so we don't lose validation records that might not have a
# matching inventory row (e.g., model retired and removed from inventory)
df_out = (df_base
          .merge(df_fsv_prep, on="_model_id", how="outer")
          .merge(df_ar_prep,  on="_model_id", how="outer")
          .merge(df_lsv_prep, on="_model_id", how="outer"))

# Ensure only requested models are in final output (outer merge can pull extras)
df_out = df_out[df_out["_model_id"].isin(model_filter_set)]

print(f"Joined output shape: {df_out.shape}")
df_out.head(3)

## 7. Build Final Output DataFrame

In [ ]:
def fmt_date(series):
    return pd.to_datetime(series, errors="coerce").dt.strftime("%d-%b-%Y").fillna("-")

def safe_col(df, col):
    """Return column if present, else a Series of '-'."""
    return df[col].fillna("-") if col in df.columns else pd.Series("-", index=df.index)

final = pd.DataFrame()

# --- Identity ---
final["Ref"]              = range(1, len(df_out) + 1)
final["Model ID"]         = df_out["_model_id"].values
final["Model Name"]       = safe_col(df_out, INV_MODEL_NAME_COL).values
final["Use Status"]       = safe_col(df_out, INV_USE_STATUS_COL).values
final["Retirement Date"]  = "-"   # Not in source — fill manually
final["Inventory Quarter"]= safe_col(df_out, "_quarter").values

# --- FSV ---
final["FSV Validation Task ID"]                          = safe_col(df_out, "fsv_task_id").values
final["FSV Date"]                                        = fmt_date(df_out.get("fsv_date", pd.Series(pd.NaT, index=df_out.index))).values
final["FSV Quarter"]                                     = safe_col(df_out, "fsv_quarter").values
final["Risk Tier Assessment Performed Within FSV (Y/N)"] = "-"
final["OGM Plan Review (Y/N) - FSV"]                    = "-"
final["FSV QA Review Performed (Y/N)"]                  = "-"

# --- AR ---
final["AR Validation Task ID"]                           = safe_col(df_out, "ar_task_id").values
final["AR Date"]                                         = fmt_date(df_out.get("ar_date", pd.Series(pd.NaT, index=df_out.index))).values
final["AR Quarter"]                                      = safe_col(df_out, "ar_quarter").values
final["Risk Tier Assessment Performed Within AR (Y/N)"]  = "-"
final["OGM Plan Review (Y/N) - AR"]                     = "-"
final["AR QA Review Performed (Y/N)"]                   = "-"

# --- LSV ---
final["LSV Validation Task ID"]                         = safe_col(df_out, "lsv_task_id").values
final["LSV Date"]                                        = fmt_date(df_out.get("lsv_date", pd.Series(pd.NaT, index=df_out.index))).values
final["LSV Quarter"]                                     = safe_col(df_out, "lsv_quarter").values
final["OGM Plan Review (Y/N) - LSV"]                    = "-"

print(f"Final output shape : {final.shape}")
final.head(5)

## 8. Write Formatted Excel Output

In [ ]:
wb = Workbook()
ws = wb.active
ws.title = "MRM Output Report"

# ---- Styles ----------------------------------------------------------------
HEADER_FILL = PatternFill("solid", fgColor="1F3864")  # navy
ID_FILL     = PatternFill("solid", fgColor="F2F2F2")  # light grey  (identity cols)
FSV_FILL    = PatternFill("solid", fgColor="D6E4F0")  # blue        (FSV block)
AR_FILL     = PatternFill("solid", fgColor="D5F0D6")  # green       (AR block)
LSV_FILL    = PatternFill("solid", fgColor="FFF2CC")  # yellow      (LSV block)

HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=10)
DATA_FONT   = Font(name="Arial", size=10)
MANUAL_FONT = Font(name="Arial", size=10, color="808080", italic=True)  # grey italic for '-' manual fields
CENTER      = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT        = Alignment(horizontal="left",   vertical="center", wrap_text=True)
thin        = Side(style="thin", color="AAAAAA")
BORDER      = Border(left=thin, right=thin, top=thin, bottom=thin)

# Map column index (1-based) → section fill
# Cols 1-6: identity  | 7-12: FSV | 13-18: AR | 19-22: LSV
def col_fill(ci):
    if  7 <= ci <= 12: return FSV_FILL
    if 13 <= ci <= 18: return AR_FILL
    if 19 <= ci <= 22: return LSV_FILL
    return ID_FILL

# ---- Section header row (row 1) -------------------------------------------
section_labels = {
    (1, 6):   ("Model Information",  "1F3864", "FFFFFF"),
    (7, 12):  ("FSV",                "1A5276", "FFFFFF"),
    (13, 18): ("AR",                 "1D6A39", "FFFFFF"),
    (19, 22): ("LSV",                "7D6608", "FFFFFF"),
}

for (start, end), (label, bg, fg) in section_labels.items():
    cell = ws.cell(row=1, column=start, value=label)
    cell.font      = Font(name="Arial", bold=True, color=fg, size=11)
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = CENTER
    cell.border    = BORDER
    if end > start:
        ws.merge_cells(start_row=1, start_column=start,
                       end_row=1,   end_column=end)

# ---- Column header row (row 2) --------------------------------------------
headers = list(final.columns)
for ci, h in enumerate(headers, start=1):
    cell = ws.cell(row=2, column=ci, value=h)
    cell.font      = HEADER_FONT
    cell.fill      = HEADER_FILL
    cell.alignment = CENTER
    cell.border    = BORDER

# ---- Data rows (start row 3) ----------------------------------------------
manual_fill_cols = {
    "Retirement Date",
    "Risk Tier Assessment Performed Within FSV (Y/N)",
    "OGM Plan Review (Y/N) - FSV",
    "FSV QA Review Performed (Y/N)",
    "Risk Tier Assessment Performed Within AR (Y/N)",
    "OGM Plan Review (Y/N) - AR",
    "AR QA Review Performed (Y/N)",
    "OGM Plan Review (Y/N) - LSV",
}
manual_col_indices = {i+1 for i, h in enumerate(headers) if h in manual_fill_cols}

for ri, row in enumerate(final.itertuples(index=False), start=3):
    for ci, val in enumerate(row, start=1):
        cell = ws.cell(row=ri, column=ci, value=val)
        cell.fill      = col_fill(ci)
        cell.alignment = CENTER if ci == 1 else LEFT
        cell.border    = BORDER
        cell.font      = MANUAL_FONT if ci in manual_col_indices else DATA_FONT

# ---- Column widths ---------------------------------------------------------
col_widths = {
     1: 6,   2: 18,  3: 40,  4: 18,  5: 16,  6: 12,   # identity
     7: 28,  8: 16,  9: 10, 10: 36, 11: 22, 12: 26,   # FSV
    13: 28, 14: 16, 15: 10, 16: 36, 17: 22, 18: 26,   # AR
    19: 28, 20: 16, 21: 10, 22: 22,                    # LSV
}
for ci, w in col_widths.items():
    ws.column_dimensions[get_column_letter(ci)].width = w

ws.row_dimensions[1].height = 22
ws.row_dimensions[2].height = 40
ws.freeze_panes = "A3"   # freeze both header rows

# ---- Summary sheet --------------------------------------------------------
ws2 = wb.create_sheet("Summary")
ws2["A1"] = "Summary"
ws2["A1"].font = Font(bold=True, size=13)

summary_rows = [
    ("Total output rows",          len(final)),
    ("Unique models in output",     final["Model ID"].nunique()),
    ("Models in filter list",       len(model_filter_set)),
    ("Models not found in files",   len(model_filter_set - set(final["Model ID"].unique()))),
    ("FSV records populated",       (final["FSV Validation Task ID"] != "-").sum()),
    ("AR records populated",        (final["AR Validation Task ID"]  != "-").sum()),
    ("LSV records populated",       (final["LSV Validation Task ID"] != "-").sum()),
]
for r, (label, val) in enumerate(summary_rows, start=3):
    ws2.cell(row=r, column=1, value=label).font = Font(name="Arial", size=10, bold=True)
    ws2.cell(row=r, column=2, value=val).font   = Font(name="Arial", size=10)
ws2.column_dimensions["A"].width = 35
ws2.column_dimensions["B"].width = 15

wb.save(OUTPUT_FILE)
print(f"\n✅  Output saved → {OUTPUT_FILE}")
print(f"    {len(final)} rows  |  {final['Model ID'].nunique()} unique models")

## 9. Optional — Preview Missing Models

In [ ]:
found_in_output = set(final["Model ID"].unique())
missing_models  = sorted(model_filter_set - found_in_output)

if missing_models:
    print(f"⚠️  {len(missing_models)} model(s) from your filter list had NO data in any quarter:")
    for m in missing_models:
        print(f"   • {m}")
else:
    print("✅  All models from the filter list were found and included in the output.")